In [2]:
import numpy as np
from IPython.display import Image, display
from sedona.spark import SedonaContext
import itertools
import os

In [3]:
additional_packages = [
    'org.apache.sedona:sedona-spark-3.5_2.12:1.7.1',
    'org.datasyslab:geotools-wrapper:1.7.1-28.5',
]

bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder().\
     config("spark.jars.packages", ",".join(additional_packages)).\
     config("spark.executor.memory", "5g").\
     config("spark.driver.memory", "5g").\
     config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"). \
     config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem").\
     config("spark.hadoop.fs.s3a.path.style.access", "true"). \
     getOrCreate()

sedona = SedonaContext.create(config)
sedona.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.sedona#sedona-spark-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-122b4d97-243b-4aa3-894c-95222a0a27b4;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-3.5_2.12;1.7.1 in central
	found org.apache.sedona#sedona-common;1.7.1 in central
	found org.apache.commons#commons-math3;3.6.1 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.wololo#jts2geojson;0.16.1 in central
	found org.locationtech.spatial4j#spatial4j;0.8 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-

In [4]:
import pyspark.sql.functions as f
from pyspark.sql import Window

def normalize_column(column_name: str, wage: float) -> callable:
    def transform(df):
        windowSpec = Window.partitionBy(f.lit(1)) 
        min_value = f.coalesce(f.min(column_name).over(windowSpec), f.lit(0))

        max_value = f.coalesce(f.max(column_name).over(windowSpec), f.lit(0))


        divider = f.when(
            f.col("min_value") == f.col("max_value"), f.col("min_value")
        ).otherwise(f.col("max_value") - f.col("min_value"))

        column_expression = (f.col(column_name) - f.col("min_value")) / divider

        return df \
            .withColumn("min_value", min_value)\
            .withColumn("max_value", max_value)\
            .withColumn("divider", divider)\
            .withColumn(column_name, column_expression)\
            .withColumn(column_name, wage * f.coalesce(f.col(column_name), f.lit(0)))\
            .drop("min_value", "max_value")

    return transform

In [4]:
# reading data

In [5]:
area = "POLYGON((20.8623619079589 52.0878295898438, 20.8623619079589 52.3485488891602, 21.2123298645019 52.3485488891602, 21.2123298645019 52.0878295898438, 20.8623619079589 52.0878295898438))"
intersects = f"ST_Intersects(geometry, ST_GeomFromText('{area}'))"
rs_intersects = f"RS_Intersects(rast, ST_Transform(ST_GeomFromText('{area}'), 'epsg:4326', 'epsg:3035'))"

places = (
    sedona
      .read
      .format("geoparquet")
      .load(f"s3a://{bucket_name}/source_data/places")
      .where(intersects)
)

buildings = (
    sedona
      .read
      .format("geoparquet")
      .load(f"s3a://{bucket_name}/source_data/buildings")
      .where(intersects)
)

buildings.cache().count()
buildings.createOrReplaceTempView("buildings")

In [6]:
fire_departments = (
sedona
  .read
  .format("geoparquet")
  .load(f"s3a://{bucket_name}/source_data/places")
  .where("categories.primary = 'fire_department'")
  .where(intersects)
)

fire_departments.cache().count()
fire_departments.createOrReplaceTempView("fire_departments")

police_department = (
    sedona
      .read
      .format("geoparquet")
      .load(f"s3a://{bucket_name}/source_data/places")
      .where("categories.primary = 'police_department'")
      .where(intersects)
)

police_department.cache().count()
police_department.createOrReplaceTempView("police_department")

In [7]:
fire_hydrants = (
    sedona
      .read
      .format("geoparquet")
      .load(f"s3a://{bucket_name}/source_data/infrastructure")
      .where("class = 'fire_hydrant'")
      .where(intersects)
)

fire_hydrants.cache().count()
fire_hydrants.createOrReplaceTempView("fire_hydrants")

In [8]:
population = (
    sedona
      .read
      .format("binaryFile")
      .load(f"s3a://{bucket_name}/source_data/world_population_raster")
      .selectExpr("RS_FromGeoTiff(content) AS rast")
      .where(rs_intersects)
      .selectExpr("Explode(RS_Tile(rast, 256, 256)) AS col")
      .selectExpr("col AS rast")
)

population.cache().count()
population.createOrReplaceTempView("population")

In [9]:
fire_risk = (
    sedona
      .read
      .format("binaryFile")
      .load(f"s3a://{bucket_name}/source_data/fire_risk")
      .selectExpr("risk", "RS_FromGeoTiff(content) AS rast")
      .where(rs_intersects)
      .selectExpr("risk", "Explode(RS_Tile(rast, 256, 256)) AS col")
      .selectExpr("risk", "col AS rast")
)

fire_risk.cache().count()
fire_risk.createOrReplaceTempView("fire_risk")

In [10]:
flood = (
    sedona
      .read
      .format("binaryFile")
      .load(f"s3a://{{bucket_name}}/source_data/flood")
      .selectExpr("rp", "RS_FromGeoTiff(content) AS rast")
      .where(rs_intersects)
      .selectExpr("rp", "Explode(RS_Tile(rast, 500, 500)) AS col")
      .selectExpr("rp", "col AS rast")
)
flood.cache().count()
flood.createOrReplaceTempView("flood")

## Population Data

In [11]:
sedona.sql(
"""
SELECT 
    b.id,
    ST_Buffer(ST_Intersection(b.geometry, RS_Envelope(rast)), -0.00001) AS geometry,
    rast
FROM buildings AS b
JOIN population AS p ON RS_Intersects(geometry, rast)

"""
).createOrReplaceTempView("population_data")

In [12]:
sedona.sql(
"""
SELECT 
    id,
    RS_ZonalStats(rast, geometry, "sum") AS population
FROM population_data
"""
).createOrReplaceTempView("building_population")

## flood risk

In [13]:
sedona.sql(
"""
SELECT 
    b.id,
    ST_Buffer(ST_Intersection(
        ST_Transform(b.geometry, 'epsg:4326', 'epsg:3035'),
        RS_Envelope(rast)
    ), -0.00001) AS geom,
    rast,
    rp
FROM buildings AS b
JOIN flood AS p ON RS_Intersects(geometry, rast)

"""
).createOrReplaceTempView("flood_data")

In [14]:
sedona.sql(
    """
    WITH flood_risk AS (
        SELECT 
            id,
            rp,
            Min(RS_ZonalStats(rast, geom, "min")) AS min_value,
            Max(RS_ZonalStats(rast, geom, "max")) AS max_value
        FROM flood_data
        GROUP BY id, rp
        Having min_value <> 'NaN' AND max_value <> 'NaN'
    )
    SELECT * FROM flood_risk
    PIVOT (
        FIRST(min_value) AS min,
        FIRST(max_value) AS max
        FOR rp IN (
            '10' AS flood_10,
            '20' AS flood_20
        )
    )

    """
).createOrReplaceTempView("flood_stats")

## fire risk

In [15]:
sedona.sql(
"""
SELECT 
    b.id,
    ST_Buffer(ST_Intersection(
        ST_Transform(b.geometry, 'epsg:4326', 'epsg:3857'),
        RS_Envelope(rast)
    ), -0.00001) AS geom,
    rast,
    risk
FROM buildings AS b
JOIN fire_risk AS p ON RS_Intersects(geometry, rast)

"""
).createOrReplaceTempView("fire_risk_data")

In [16]:
sedona.sql(
    """
    WITH fire_risk AS (
        SELECT 
            id,
            risk,
            Min(RS_ZonalStats(rast, geom, "min")) AS min_value,
            Max(RS_ZonalStats(rast, geom, "max")) AS max_value
        FROM fire_risk_data
        GROUP BY id, risk
        Having min_value <> 'NaN' AND max_value <> 'NaN'
    )
    SELECT * FROM fire_risk
    PIVOT (
        FIRST(min_value) AS min,
        FIRST(max_value) AS max
        FOR risk IN (
            'high' AS fire_risk_high,
            'intermediate' AS fire_risk_intermediate,
            'low' AS fire_risk_low
        )
    )

    """
).createOrReplaceTempView("fire_risk_stats")

In [17]:
## buildings nearby

In [18]:
sedona.sql(
    """
    WITH nearby_buildings AS (
       SELECT
            b1.id AS b1_id,
            b2.id AS b2_id
        FROM buildings AS b1
        JOIN buildings AS b2 ON ST_DWithin(b1.geometry, b2.geometry, 500, true) 
    )
    SELECT b1_id AS id, count(*) AS density FROM nearby_buildings
    GROUP BY b1_id

    """
).createOrReplaceTempView("building_density")

In [19]:
## closest fire station distance

In [20]:
sedona.sql(
    """
    SELECT 
        b.id,
        ST_DistanceSpheroid(b.geometry, f.geometry) AS distance
    FROM buildings AS b
    JOIN fire_departments AS f ON ST_KNN(b.geometry, f.geometry, 1)
    """
).createOrReplaceTempView("closest_fire_department")

In [21]:
## closest police station distance

In [22]:
sedona.sql(
    """
    SELECT 
        b.id,
        ST_DistanceSpheroid(b.geometry, p.geometry) AS distance
    FROM buildings AS b
    JOIN police_department AS p ON ST_KNN(b.geometry, p.geometry, 1)
    """
).createOrReplaceTempView("closest_police_department")

In [23]:
## closest hydrant distance

In [24]:
sedona.sql(
    """
    SELECT 
        b.id,
        ST_DistanceSpheroid(b.geometry, f.geometry) AS distance
    FROM buildings AS b
    JOIN fire_hydrants AS f ON ST_KNN(b.geometry, f.geometry, 1)
    """
).createOrReplaceTempView("closest_fire_hydrants")

In [25]:
## index

In [26]:
wages = {
    "population": 0.05,
    
    "flood_10_min": 0.1,
    "flood_10_max": 0.12,
    "flood_20_min": 0.1,
    "flood_20_max": 0.05,
    
    "fire_risk_high_min": 0.1,
    "fire_risk_high_max": 0.2,
    "fire_risk_intermediate_min": 0.03,
    "fire_risk_intermediate_max": 0.03,
    "fire_risk_low_min": 0.01,
    "fire_risk_low_max": 0.01,
    
    "density": 0.05,
    
    "closest_fire_department_distance": 0.05,
    "closest_police_department_distance": 0.05,
    "closest_fire_hydrants_distance": 0.05,
}


In [27]:
result = (sedona.sql(
    """
    SELECT 
        b.id,
        bp.population,
        f.flood_10_min,
        f.flood_10_max,
        f.flood_20_min,
        f.flood_20_max,
        fr.fire_risk_high_min,
        fr.fire_risk_high_max,
        fr.fire_risk_intermediate_min,
        fr.fire_risk_intermediate_max,
        fr.fire_risk_low_min,
        fr.fire_risk_low_max,
        bd.density,
        cf.distance AS closest_fire_department_distance,
        cp.distance AS closest_police_department_distance,
        ch.distance AS closest_fire_hydrants_distance
    FROM buildings AS b
    LEFT JOIN building_population AS bp ON bp.id = b.id
    LEFT JOIN flood_stats AS f ON f.id = b.id
    LEFT JOIN fire_risk_stats AS fr ON fr.id = b.id
    LEFT JOIN building_density AS bd ON bd.id = b.id
    LEFT JOIN closest_fire_department AS cf ON cf.id = b.id
    LEFT JOIN closest_police_department AS cp ON cp.id = b.id
    LEFT JOIN closest_fire_hydrants AS ch ON ch.id = b.id
    """
)
.transform(normalize_column("population", wages["population"]))
.transform(normalize_column("flood_10_min", wages["flood_10_min"]))
.transform(normalize_column("flood_10_max", wages["flood_10_max"]))
.transform(normalize_column("flood_20_min", wages["flood_20_min"]))
.transform(normalize_column("flood_20_max", wages["flood_20_max"]))
.transform(normalize_column("fire_risk_high_min", wages["fire_risk_high_min"]))
.transform(normalize_column("fire_risk_high_max", wages["fire_risk_high_max"]))
.transform(normalize_column("fire_risk_intermediate_min", wages["fire_risk_intermediate_min"]))
.transform(normalize_column("fire_risk_intermediate_max", wages["fire_risk_intermediate_max"]))
.transform(normalize_column("fire_risk_low_min", wages["fire_risk_low_min"]))
.transform(normalize_column("fire_risk_low_max", wages["fire_risk_low_max"]))
.transform(normalize_column("density", wages["density"]))
.transform(normalize_column("closest_fire_department_distance", wages["closest_fire_department_distance"]))
.transform(normalize_column("closest_police_department_distance", wages["closest_police_department_distance"]))
.transform(normalize_column("closest_fire_hydrants_distance", wages["closest_fire_hydrants_distance"]))
).cache()

result.count()

146136

In [28]:
result.selectExpr(
    "id",
    """
    (population + flood_10_min + flood_10_max + flood_20_min + flood_20_max + fire_risk_high_min + 
    fire_risk_high_max + fire_risk_intermediate_min + fire_risk_intermediate_max + fire_risk_low_min +
    fire_risk_low_max + closest_fire_department_distance + closest_police_department_distance + closest_fire_hydrants_distance) AS index
    """
).createOrReplaceTempView("index")

In [29]:
viz_area = "POLYGON((21.003947 52.225983, 21.048335 52.225983, 21.048335 52.249204, 21.003947 52.249204, 21.003947 52.225983))"
df = sedona.sql(
    """
    SELECT 
        index.id,
        index.index,
        b.geometry
    FROM index
    JOIN buildings AS b ON b.id = index.id
    """
)

In [30]:
df.cache().count()

146136

In [31]:
from sedona.spark import SedonaKepler
SedonaKepler.create_map(df, "map")

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(data={'map': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21,…